# Store 014: OWL and SHACL packs, Qwen on a free Colab GPU
Run with **Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**. It loads Store 014 into Postgres, starts Qwen2.5 7B in Ollama, and gives it each OWL/SHACL pack and four questions about 5 kg atta. The answers are saved to `/content/results` and zipped at the end. No keys are needed.

In [ ]:
!apt-get -qq update && apt-get -qq install -y postgresql > /dev/null
!service postgresql start
!sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'postgres'"
!rm -rf /content/ds && git clone -q --depth 1 -b owl-packs-test https://github.com/kvinayakpai/domain-series.git /content/ds
!pip install -q requests psycopg2-binary

In [ ]:
%cd /content/ds
!python3 auto-ontology-demo/seed_store014.py > /tmp/store014.sql
!PGPASSWORD=postgres psql -h localhost -U postgres -q -c "CREATE DATABASE store014"
!PGPASSWORD=postgres psql -h localhost -U postgres -d store014 -v ON_ERROR_STOP=1 -q -f /tmp/store014.sql
!PGPASSWORD=postgres psql -h localhost -U postgres -d store014 -Atc "select count(*) from store.pos_lines"

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(['ollama','serve'], stdout=open('/tmp/ollama.log','w'), stderr=subprocess.STDOUT)
time.sleep(8)
!ollama pull qwen2.5:7b-instruct-q4_K_M
!printf 'FROM qwen2.5:7b-instruct-q4_K_M\nPARAMETER num_ctx 16384\nPARAMETER temperature 0.2\n' > /tmp/Modelfile && ollama create qwen-16k -f /tmp/Modelfile
!ollama list

In [ ]:
%cd /content/ds
!PGHOST=localhost PGUSER=postgres PGPASSWORD=postgres python3 owl-packs-test/run.py /content/results qwen

In [ ]:
import json, glob
for f in sorted(glob.glob('/content/results/*.json')):
    r = json.load(open(f))
    print('=====', f.split('/')[-1], 'seconds:', r.get('seconds'), r.get('error', ''))
    print(r.get('final', ''))
